7 mins 11.2 secs

## Load libraries

In [17]:
import math
import numpy as np
import pandas as pd
from collections import defaultdict

from statsmodels.tsa.seasonal import STL

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import ParameterGrid

import torch
import torch.nn as nn
from tqdm import tqdm

## Config

In [18]:

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")

# rolling CV
ROLLING_TRAIN_WINDOW = 120   # 10 years
ROLLING_VAL_WINDOW   = 12    # 1 year

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---- feature lists (adjust to match your data) ----
continuous_cols = [
    "AverageNeighbourPrice",
    "local_I",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
]

categorical_cols = [
    "LMIQuadrant__2",
    "LMIQuadrant__3",
    "LMIQuadrant__4",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

# ---- feature groups for composite kernel ----
spatial_cont = ["centroid_x", "centroid_y", "CoL_distance_km"]

local_struct = [
    "AverageNeighbourPrice",
    "local_I",
    "area_km2",
    "LA_FE",
    "dwelling_stock",
    "population",
    "rail_station_entry_exit",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
]

macro = [
    "sdlt_perc_threshold",
    "ashe_weekly",
    "base_rate",
    "GDP",
    "CPIH",
]

def get_indices_for_groups(all_feature_cols, lag_cols):
    """Return index arrays for each group given the full feature column order."""
    def idx(names):
        return [all_feature_cols.index(c) for c in names if c in all_feature_cols]

    spatial_idx = idx(spatial_cont)
    local_idx   = idx(local_struct)
    macro_idx   = idx(macro)
    cat_idx     = idx(categorical_cols)
    lag_idx     = [i for i, c in enumerate(all_feature_cols) if c.startswith("stl_")]

    return spatial_idx, local_idx, macro_idx, cat_idx, lag_idx

# ---- lag plan (always 1 & 12; variants with 2–6; optional 24) ----
lag_combinations = [
    [1, 12],
    [1, 2, 12],
    [1, 2, 3, 12],
    [1, 2, 3, 4, 5, 6, 12],
    [1, 12, 24],
    [1, 2, 12, 24],
    [1, 2, 3, 12, 24],
    [1, 2, 3, 4, 5, 6, 12, 24],
]
all_lags = sorted({l for combo in lag_combinations for l in combo})

# -----------------------
# "Kernel-style" model hyperparams
# -----------------------
rff_param_grid = {
    "hidden_dim":    [64, 128],
    "gamma_spatial": [0.1, 0.5, 1.0],
    "gamma_local":   [0.1, 0.5, 1.0],
    "gamma_macro":   [0.05, 0.1, 0.2],
    "gamma_lag":     [0.1, 0.5, 1.0],
    "weight_decay":  [0.0, 1e-4],
}

# RFF output dimensions per feature group
RFF_DIMS = {
    "spatial": 128,   # was 64
    "local":   256,   # was 128
    "macro":   64,    # was 32
    "lag":     512,  # was 256
}


Using device: cuda


## Kernel

In [19]:
class RFFBlock(nn.Module):
    """
    Random Fourier Features block:
      x -> sqrt(2/m) * cos(Wx + b)
    where W ~ N(0, 2 * gamma), b ~ Uniform(0, 2π)
    """
    def __init__(self, input_dim, output_dim, gamma):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.gamma = gamma

        self.W = nn.Parameter(
            torch.randn(output_dim, input_dim) * math.sqrt(2 * gamma),
            requires_grad=False
        )
        self.b = nn.Parameter(
            torch.rand(output_dim) * 2 * math.pi,
            requires_grad=False
        )
        self.scale = math.sqrt(2.0 / output_dim)

    def forward(self, x):
        return torch.cos(x @ self.W.T + self.b) * self.scale

class CompositeRFF(nn.Module):
    def __init__(self, group_dims, output_dims, gammas):
        """
        group_dims: list of input dims per group
        output_dims: list of RFF dims per group
        gammas: list of gamma values per group
        """
        super().__init__()
        self.blocks = nn.ModuleList([
            RFFBlock(gdim, odim, gamma)
            for gdim, odim, gamma in zip(group_dims, output_dims, gammas)
        ])

    def forward(self, groups):
        # groups: list of tensors [batch, group_dim]
        mapped = [block(g) for block, g in zip(self.blocks, groups)]
        return torch.cat(mapped, dim=1)

class RFFKernelRegressor(nn.Module):
    def __init__(self, group_dims, output_dims, gammas, hidden_dim=128):
        super().__init__()
        self.rff = CompositeRFF(group_dims, output_dims, gammas)
        total_rff_dim = sum(output_dims)

        self.mlp = nn.Sequential(
            nn.Linear(total_rff_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, groups):
        z = self.rff(groups)
        return self.mlp(z).squeeze(-1)

def train_rff_model(
    model,
    train_groups,
    y_train_scaled,
    val_groups,
    y_val_scaled,
    epochs=50,
    lr=1e-3,
    weight_decay=0.0,
    patience=5,
):
    """
    Train RFFKernelRegressor with MSE loss + early stopping.

    weight_decay: L2 regularisation (SVR-like 'C' effect).
    patience: stop if val loss doesn't improve for this many epochs.
    """
    device = next(model.parameters()).device
    y_train = torch.tensor(y_train_scaled, dtype=torch.float32, device=device)
    y_val   = torch.tensor(y_val_scaled,   dtype=torch.float32, device=device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay  # L2 regularisation
    )
    loss_fn = nn.MSELoss()

    best_val_loss = float("inf")
    best_val_pred = None
    epochs_no_improve = 0

    for _ in range(epochs):
        # ---- train step ----
        model.train()
        optimizer.zero_grad()
        y_pred_train = model(train_groups)
        loss = loss_fn(y_pred_train, y_train)
        loss.backward()
        optimizer.step()

        # ---- validation step ----
        model.eval()
        with torch.no_grad():
            y_pred_val = model(val_groups)
            val_loss = loss_fn(y_pred_val, y_val).item()

        if val_loss < best_val_loss - 1e-5:  # small tolerance
            best_val_loss = val_loss
            best_val_pred = y_pred_val.detach().cpu().numpy()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    # use best_val_pred (early-stopped)
    return best_val_pred


## Evaluation metric functions

In [20]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.sqrt(np.mean((y - yhat) ** 2))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return np.mean(np.abs(y - yhat)) / scale


## Load data

In [21]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.sort_values([ENTITY_COL, TIME_COL]).reset_index(drop=True)

mask_tv = (df[TIME_COL] >= TRAIN_START_DATE) & (df[TIME_COL] <= TRAIN_END_DATE)
df_tv = df.loc[mask_tv].copy()

# all unique months in train+val window
dates = sorted(df_tv[TIME_COL].unique())

## Helper for feature group indices (per lag_set)

In [22]:
def build_group_indices(all_feature_cols, lag_cols):
    def idx(names):
        return [all_feature_cols.index(c) for c in names if c in all_feature_cols]

    spatial_idx = idx(spatial_cont)
    local_idx   = idx(local_struct)
    macro_idx   = idx(macro)
    lag_idx     = [i for i, c in enumerate(all_feature_cols) if c.startswith("stl_")]

    return spatial_idx, local_idx, macro_idx, lag_idx

def build_groups_tensors(X, group_indices):
    X_t = torch.tensor(X.values, dtype=torch.float32, device=device)
    groups = [X_t[:, idx] for idx in group_indices]
    return groups

## Training with STL + Rolling CV

In [23]:
# =========================================================
# CV STRUCTURE:
#   outer loop   = folds (STL + all lags computed once per fold)
#   mid loop     = lag_set (subselect lag columns, scale, etc.)
#   inner loop   = RF params (fit, predict, metrics)
# =========================================================

# metrics_store[(lag_tuple, params_key)] = dict of metric lists across folds
metrics_store = {}

# Pre-build list of folds based on dates
fold_specs = []
start_idx = ROLLING_TRAIN_WINDOW
while True:
    train_end_idx = start_idx
    val_start_idx = train_end_idx
    val_end_idx   = val_start_idx + ROLLING_VAL_WINDOW
    if val_end_idx > len(dates):
        break

    train_start = dates[train_end_idx - ROLLING_TRAIN_WINDOW]
    train_end   = dates[train_end_idx - 1]
    val_start   = dates[val_start_idx]
    val_end     = dates[val_end_idx - 1]

    fold_specs.append((train_start, train_end, val_start, val_end))
    start_idx += ROLLING_VAL_WINDOW

print(f"Number of folds: {len(fold_specs)}")

# =========================================================
# MAIN LOOP: FOLDS → (lag_set → params)
# =========================================================
for fold_no, (train_start, train_end, val_start, val_end) in enumerate(fold_specs, start=1):
    print(f"\n=== Fold {fold_no}: "
          f"Train {train_start:%Y-%m}–{train_end:%Y-%m}, "
          f"Val {val_start:%Y-%m}–{val_end:%Y-%m} ===")

    # ---- slice fold train/val ----
    mask_train = (df_tv[TIME_COL] >= train_start) & (df_tv[TIME_COL] <= train_end)
    mask_val   = (df_tv[TIME_COL] >= val_start)   & (df_tv[TIME_COL] <= val_end)

    fold_train = df_tv.loc[mask_train].copy()
    fold_val   = df_tv.loc[mask_val].copy()

    # ---- STL on training ONLY (per LA) ----
    fold_train["stl_trend"]    = np.nan
    fold_train["stl_seasonal"] = np.nan
    fold_train["stl_resid"]    = np.nan

    for la, sub in tqdm(fold_train.groupby(ENTITY_COL),
                        desc=f"  STL (train only), fold {fold_no}", leave=False):
        sub = sub.sort_values(TIME_COL)
        series = sub[TARGET_COL].astype(float)
        if len(series) < 24:  # too short for STL, skip
            continue
        stl = STL(series, period=12, robust=True)
        res = stl.fit()
        fold_train.loc[sub.index, "stl_trend"]    = res.trend
        fold_train.loc[sub.index, "stl_seasonal"] = res.seasonal
        fold_train.loc[sub.index, "stl_resid"]    = res.resid

    # ---- extend STL into validation (per LA) ----
    fold_val["stl_trend"]    = np.nan
    fold_val["stl_seasonal"] = np.nan
    fold_val["stl_resid"]    = 0.0  # unknown future residuals

    for la in fold_train[ENTITY_COL].unique():
        sub_train = fold_train[fold_train[ENTITY_COL] == la].sort_values(TIME_COL)
        sub_val   = fold_val[fold_val[ENTITY_COL] == la].sort_values(TIME_COL)

        if sub_val.empty or sub_train["stl_trend"].isna().all():
            continue

        n_future = len(sub_val)

        # seasonal: repeat last 12-month pattern (or fewer if early)
        season_train = sub_train["stl_seasonal"].dropna().values
        if season_train.size == 0:
            continue
        if len(season_train) >= 12:
            base_pattern = season_train[-12:]
        else:
            base_pattern = season_train
        reps = int(np.ceil(n_future / len(base_pattern)))
        season_future = np.tile(base_pattern, reps)[:n_future]

        # trend: simple linear extrapolation
        trend_train = sub_train["stl_trend"].dropna().values
        t_idx = np.arange(len(trend_train))
        if len(trend_train) >= 2:
            coef = np.polyfit(t_idx, trend_train, 1)
            future_t = np.arange(len(trend_train), len(trend_train) + n_future)
            trend_future = coef[0] * future_t + coef[1]
        else:
            trend_future = np.full(n_future, trend_train[-1])

        future_idx = sub_val.index
        fold_val.loc[future_idx, "stl_trend"]    = trend_future
        fold_val.loc[future_idx, "stl_seasonal"] = season_future

    # ---- combine train+val to compute ALL LAG COLS (all_lags) once per fold ----
    fold_train["is_train"] = True
    fold_val["is_train"]   = False

    combined = pd.concat([fold_train, fold_val], axis=0)
    combined = combined.sort_values([ENTITY_COL, TIME_COL])

    # create lag columns for ALL lags (superset) on STL components
    for lag in all_lags:
        for comp in ["stl_trend", "stl_seasonal", "stl_resid"]:
            col = f"{comp}_lag{lag}"
            combined[col] = combined.groupby(ENTITY_COL)[comp].shift(lag)

    # ---------------------------------------------------------
    # For this fold: loop over lag_set, then RF params
    # Using the precomputed lag columns in `combined`
    # ---------------------------------------------------------
    for lag_set in lag_combinations:
        print(f"  Lag set: {lag_set}")
        lag_cols = [   
            f"{comp}_lag{lag}"
            for comp in ["stl_trend", "stl_seasonal", "stl_resid"]
            for lag in lag_set
        ]

        # split back into train/val
        fold_train_lag = combined[combined["is_train"]].copy()
        fold_val_lag   = combined[~combined["is_train"]].copy()

        # require ALL FEATURES (continuous + categorical + lags) present
        full_feature_cols = continuous_cols + categorical_cols + lag_cols
        fold_train_lag = fold_train_lag.dropna(subset=full_feature_cols)
        fold_val_lag   = fold_val_lag.dropna(subset=full_feature_cols)

        if fold_train_lag.empty or fold_val_lag.empty:
            print("    (skip: no data after feature drop)")
            continue

        # ----------------------
        # BUILD X, y AFTER NaN DROP
        # ----------------------
        X_train = fold_train_lag[continuous_cols + categorical_cols + lag_cols].copy()
        y_train = fold_train_lag[TARGET_COL].values

        X_val   = fold_val_lag[continuous_cols + categorical_cols + lag_cols].copy()
        y_val   = fold_val_lag[TARGET_COL].values

        # ----------------------
        # SCALE X (TRAIN ONLY)
        # ----------------------
        scale_cols = continuous_cols + lag_cols
        scaler = StandardScaler()
        X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
        X_val[scale_cols]   = scaler.transform(X_val[scale_cols])

        # ----------------------
        # SCALE y ONCE PER (FOLD, LAG_SET)
        # ----------------------
        y_scaler = StandardScaler()
        y_train_raw = y_train.reshape(-1, 1)
        y_val_raw   = y_val.reshape(-1, 1)

        y_train_scaled = y_scaler.fit_transform(y_train_raw).ravel()
        y_val_scaled   = y_scaler.transform(y_val_raw).ravel()

        
        # group indices for this lag_set
        spatial_idx, local_idx, macro_idx, lag_idx = build_group_indices(
            full_feature_cols, lag_cols
        )
        group_indices = [spatial_idx, local_idx, macro_idx, lag_idx]

        # tensor groups (on device)
        train_groups = build_groups_tensors(X_train, group_indices)
        val_groups   = build_groups_tensors(X_val,   group_indices)

        # inner loop over SVM hyperparameters
        for params in ParameterGrid(rff_param_grid):
            print(f"  RFF params: {params}")
            # make a hashable key for this (lag_set, params)
            lag_key = tuple(lag_set)
            params_key = tuple(sorted(params.items()))
            key = (lag_key, params_key)

            if key not in metrics_store:
                metrics_store[key] = {
                    "lag_set": lag_key,
                    "params": params,
                    "mae": [],
                    "rmse": [],
                    "smape": [],
                    "mase": [],
                    "folds": 0,
                }

            group_dims = [len(idx) for idx in group_indices]
            output_dims = [
                RFF_DIMS["spatial"],
                RFF_DIMS["local"],
                RFF_DIMS["macro"],
                RFF_DIMS["lag"],
            ]
            gammas = [
                params["gamma_spatial"],  # spatial
                params["gamma_local"],    # local
                params["gamma_macro"],    # macro
                params["gamma_lag"],      # lags
            ]

            model = RFFKernelRegressor(
                group_dims=group_dims,
                output_dims=output_dims,
                gammas=gammas,
                hidden_dim=params["hidden_dim"],
            ).to(device)

            y_pred_scaled = train_rff_model(
                model,
                train_groups,
                y_train_scaled,
                val_groups,
                y_val_scaled,
                epochs=50,
                lr=1e-3,
                weight_decay=params["weight_decay"],
                patience=5,
            )

            # back to £ scale
            y_pred = y_scaler.inverse_transform(
                y_pred_scaled.reshape(-1, 1)
            ).ravel()

            y_val_true   = y_val_raw.ravel()
            y_train_true = y_train_raw.ravel()

            metrics_store[key]["mae"].append(mae(y_val_true, y_pred))
            metrics_store[key]["rmse"].append(rmse(y_val_true, y_pred))
            metrics_store[key]["smape"].append(smape(y_val_true, y_pred))
            metrics_store[key]["mase"].append(mase(y_val_true, y_pred, y_train_true))
            metrics_store[key]["folds"] += 1

# =========================================================
# AGGREGATE RESULTS OVER FOLDS
# =========================================================
rows = []
for key, val in metrics_store.items():
    if val["folds"] == 0:
        continue
    rows.append({
        "model_type": "SVR_CompositeKernel",
        "lag_set": val["lag_set"],
        "params": val["params"],
        "folds": val["folds"],
        "MAE_mean":   float(np.mean(val["mae"])),
        "MAE_std":    float(np.std(val["mae"])),
        "RMSE_mean":  float(np.mean(val["rmse"])),
        "RMSE_std":   float(np.std(val["rmse"])),
        "sMAPE_mean": float(np.mean(val["smape"])),
        "sMAPE_std":  float(np.std(val["smape"])),
        "MASE_mean":  float(np.mean(val["mase"])),
        "MASE_std":   float(np.std(val["mase"])),
    })



Number of folds: 5

=== Fold 1: Train 2007-04–2017-03, Val 2017-04–2018-03 ===


  Lag set: [1, 12]
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 64, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 64, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 128, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 128, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 64, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 64, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 128, 'weight_decay': 0.0}
  RFF params: {'gamma_la

  Lag set: [1, 12]
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 64, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 64, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 128, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 128, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 64, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 64, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 128, 'weight_decay': 0.0}
  RFF params: {'gamma_la

  Lag set: [1, 12]
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 64, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 64, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 128, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 128, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 64, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 64, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 128, 'weight_decay': 0.0}
  RFF params: {'gamma_la

  Lag set: [1, 12]
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 64, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 64, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 128, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 128, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 64, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 64, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 128, 'weight_decay': 0.0}
  RFF params: {'gamma_la

  Lag set: [1, 12]
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 64, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 64, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 128, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.1, 'hidden_dim': 128, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 64, 'weight_decay': 0.0}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 64, 'weight_decay': 0.0001}
  RFF params: {'gamma_lag': 0.1, 'gamma_local': 0.1, 'gamma_macro': 0.05, 'gamma_spatial': 0.5, 'hidden_dim': 128, 'weight_decay': 0.0}
  RFF params: {'gamma_la

## Results

In [26]:
results_df = pd.DataFrame(rows).sort_values("RMSE_mean").reset_index(drop=True)
print(results_df.head(20))
results_df.to_csv("../../results/svm_rffapprox_leakfree_stl_rollingcv_results.csv", index=False)

             model_type                     lag_set  \
0   SVR_CompositeKernel               (1, 2, 3, 12)   
1   SVR_CompositeKernel               (1, 2, 3, 12)   
2   SVR_CompositeKernel      (1, 2, 3, 4, 5, 6, 12)   
3   SVR_CompositeKernel                     (1, 12)   
4   SVR_CompositeKernel                     (1, 12)   
5   SVR_CompositeKernel              (1, 2, 12, 24)   
6   SVR_CompositeKernel      (1, 2, 3, 4, 5, 6, 12)   
7   SVR_CompositeKernel  (1, 2, 3, 4, 5, 6, 12, 24)   
8   SVR_CompositeKernel               (1, 2, 3, 12)   
9   SVR_CompositeKernel               (1, 2, 3, 12)   
10  SVR_CompositeKernel  (1, 2, 3, 4, 5, 6, 12, 24)   
11  SVR_CompositeKernel           (1, 2, 3, 12, 24)   
12  SVR_CompositeKernel                     (1, 12)   
13  SVR_CompositeKernel               (1, 2, 3, 12)   
14  SVR_CompositeKernel               (1, 2, 3, 12)   
15  SVR_CompositeKernel                  (1, 2, 12)   
16  SVR_CompositeKernel              (1, 2, 12, 24)   
17  SVR_Co

In [25]:
print(results_df.head(20))

             model_type                     lag_set  \
0   SVR_CompositeKernel               (1, 2, 3, 12)   
1   SVR_CompositeKernel               (1, 2, 3, 12)   
2   SVR_CompositeKernel      (1, 2, 3, 4, 5, 6, 12)   
3   SVR_CompositeKernel                     (1, 12)   
4   SVR_CompositeKernel                     (1, 12)   
5   SVR_CompositeKernel              (1, 2, 12, 24)   
6   SVR_CompositeKernel      (1, 2, 3, 4, 5, 6, 12)   
7   SVR_CompositeKernel  (1, 2, 3, 4, 5, 6, 12, 24)   
8   SVR_CompositeKernel               (1, 2, 3, 12)   
9   SVR_CompositeKernel               (1, 2, 3, 12)   
10  SVR_CompositeKernel  (1, 2, 3, 4, 5, 6, 12, 24)   
11  SVR_CompositeKernel           (1, 2, 3, 12, 24)   
12  SVR_CompositeKernel                     (1, 12)   
13  SVR_CompositeKernel               (1, 2, 3, 12)   
14  SVR_CompositeKernel               (1, 2, 3, 12)   
15  SVR_CompositeKernel                  (1, 2, 12)   
16  SVR_CompositeKernel              (1, 2, 12, 24)   
17  SVR_Co